In [5]:
import torch.optim as optim

from src.load_and_save import save_model
from src.training import get_accuracy
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch

In [6]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Custom transform to one-hot encode the labels
class OneHotEncode:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def __call__(self, label):
        return torch.eye(self.num_classes)[label]

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

In [7]:
from src.improved_model import BinarizingCNN

# Instantiate the model
model = BinarizingCNN().to(device)
model.set_scramble_distance(0.05)
#criterion = nn.CrossEntropyLoss()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=float(1e-2))

In [8]:
from src.training import get_average_separation

# Training loop
num_epochs: int = 500
target_accuracy: float = .95
maximum_scramble_distance: float = 5.0

test_data, _ = next(iter(test_dataloader))
test_data.to(device)


for epoch in range(num_epochs):
    for inputs, labels in train_dataloader:
        if epoch == 0:
            break
        # Forward pass
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.layer2.bias.data.clamp_(min=1.0)
        model.layer3.bias.data.clamp_(min=1.0)
        #model.layer2.bias.data.clamp_(min=-2.19722)
    # separation: float = get_average_separation(test_data, model)

    # Assess progress:
    # data: np.ndarray = get_intermediate_outputs_as_numpy(model, train_dataloader)
    validation_accuracy: float = get_accuracy(model, val_dataloader)
    if validation_accuracy > target_accuracy:
        model.set_scramble_distance(min(model.scramble_distance + .1, maximum_scramble_distance))
        print(f"Scramble distance: {model.scramble_distance:.2f}")
    print(f'Epoch [{epoch+1}/{num_epochs}], Accuracy: {validation_accuracy:.4f}')


Epoch [1/500], Accuracy: 0.0892
Epoch [2/500], Accuracy: 0.8939
Epoch [3/500], Accuracy: 0.9289
Epoch [4/500], Accuracy: 0.9423
Epoch [5/500], Accuracy: 0.9487
Scramble distance: 0.15
Epoch [6/500], Accuracy: 0.9504
Epoch [7/500], Accuracy: 0.9461
Epoch [8/500], Accuracy: 0.9487
Scramble distance: 0.25
Epoch [9/500], Accuracy: 0.9524
Epoch [10/500], Accuracy: 0.9408
Epoch [11/500], Accuracy: 0.9462
Epoch [12/500], Accuracy: 0.9494
Epoch [13/500], Accuracy: 0.9478
Scramble distance: 0.35
Epoch [14/500], Accuracy: 0.9529
Epoch [15/500], Accuracy: 0.9436
Epoch [16/500], Accuracy: 0.9497
Epoch [17/500], Accuracy: 0.9483
Epoch [18/500], Accuracy: 0.9453
Epoch [19/500], Accuracy: 0.9492
Scramble distance: 0.45
Epoch [20/500], Accuracy: 0.9527
Epoch [21/500], Accuracy: 0.9445
Epoch [22/500], Accuracy: 0.9462
Epoch [23/500], Accuracy: 0.9486
Epoch [24/500], Accuracy: 0.9492
Scramble distance: 0.55
Epoch [25/500], Accuracy: 0.9502
Epoch [26/500], Accuracy: 0.9433
Epoch [27/500], Accuracy: 0.947

KeyboardInterrupt: 

In [9]:
from src.improved_model import BinarizingNetwork


def get_signed_accuracy(model: BinarizingNetwork, dataloader: DataLoader) -> float:
    model.eval_mode()

    with torch.no_grad():  # Disable gradient computation
        all_correct: int = 0
        for inputs, labels in dataloader:
            # Move inputs and labels to the specified device
            inputs, labels = inputs.to(device), labels.to(device)
            outputs: torch.Tensor = model(inputs)
            comparison: torch.Tensor = torch.argmax(outputs, axis=1) == torch.argmax(
                labels, axis=1
            )
            all_correct += sum(comparison)
        accuracy: float = all_correct / len(dataloader.dataset)
    model.train_mode()
    return accuracy
test_data = test_data.to(device)
model.eval_mode()
model(test_data[0:1])

d = test_data[0:1]
#print(model.float_to_binary_layer(d))

print(model.second_layer(model.float_to_binary_layer(d)))
#
# # model(test_data[0,...])
print(get_signed_accuracy(model, val_dataloader))


tensor([[ 1.,  1.,  1., -1.,  1.,  1., -1., -1., -1.,  1.,  1.,  1.,  1.,  1.,
          1., -1., -1., -1.,  1.,  1.,  1., -1.,  1.,  1.,  1.,  1.,  1.,  1.,
          1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1., -1., -1.,  1., -1.,
          1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,
         -1.,  1.,  1., -1.,  1.,  1.,  1.,  1.,  1., -1.,  1.,  1., -1.,  1.,
          1.,  1.,  1.,  1.,  1.,  1.,  1.]], device='cuda:0',
       grad_fn=<SignBackward0>)
tensor(0.9192, device='cuda:0')


In [10]:
from torch import softmax

model.eval_mode()
model.layer2.bias
#1 + softmax(model.layer2.bias, dim=0)

Parameter containing:
tensor([3.5579, 3.7151, 3.5012, 1.6117, 1.8762, 1.1311, 1.2187, 1.0080, 1.2166,
        1.2233, 3.1612, 3.5776, 3.6541, 1.1799, 2.8980, 3.3143, 2.9908, 1.0243,
        4.0283, 3.7748, 3.2350, 1.3466, 2.9761, 1.2125, 1.4341, 1.2316, 3.5730,
        3.5492, 3.3030, 1.4650, 1.1516, 3.4523, 1.1210, 2.3967, 3.1849, 1.0245,
        2.6062, 1.0312, 1.0464, 1.2643, 1.7882, 1.4444, 1.1849, 2.2475, 3.7073,
        1.1186, 1.0435, 3.9899, 2.4857, 1.0574, 2.8670, 3.4544, 1.1704, 1.3532,
        1.0055, 1.2132, 1.1886, 5.3536, 1.0785, 3.7698, 3.6585, 1.5462, 1.3456,
        1.1088, 1.3134, 1.8635, 1.1846, 1.2837, 4.3617, 3.2203, 3.5909, 3.2441,
        1.2966, 1.2681, 3.3181, 1.2108, 1.8885], device='cuda:0',
       requires_grad=True)

In [12]:
save_model(model, "convnet_v2")

In [13]:
model.layer3

BinarizingLinear(in_features=77, out_features=10, bias=True)